# Spot Summary Prediction

## Imort Libraries

In [1]:
import os
import polars as pl
from datetime import datetime
from pathlib import Path

## Define Setting Configuration

### Google Authentication

In [2]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/workspace/.secrets/keita-masui-firstproject-46adeb7731b9.json"

In [3]:
bucket_uri = "gs://nexsol-data-lake-stg-usc1/raw/jepx/spot/"

## Data Loading

### Check parquet files

In [4]:
error_files = []
ok_files = []

dfs: list[pl.DataFrame] = []

for year in range(2005,2026):

    name = str(year)

    print(f"=== 読み込みテスト中: spot_summary_{str(year)}.parquet ===")
    path = f"{bucket_uri}spot_summary_{str(year)}.parquet"
    try:
        # 必要なら n_rows=100 とかで軽くテストにしても良い
        df = pl.read_parquet(path)
        dfs.append(df)
        print(f"  -> OK, shape={df.shape}")

    except Exception as e:
        error_files.append((path, str(e)))
        print(f"  -> ERROR: {e}")

print("\n==== 読み込み成功ファイル数:", len(ok_files))
print("==== 読み込み失敗ファイル数:", len(error_files))

df_spot = pl.concat(dfs, how="vertical_relaxed")

df_spot.head()

=== 読み込みテスト中: spot_summary_2005.parquet ===
  -> OK, shape=(17472, 19)
=== 読み込みテスト中: spot_summary_2006.parquet ===
  -> OK, shape=(17520, 19)
=== 読み込みテスト中: spot_summary_2007.parquet ===
  -> OK, shape=(17568, 19)
=== 読み込みテスト中: spot_summary_2008.parquet ===
  -> OK, shape=(17520, 19)
=== 読み込みテスト中: spot_summary_2009.parquet ===
  -> OK, shape=(17520, 19)
=== 読み込みテスト中: spot_summary_2010.parquet ===
  -> OK, shape=(17520, 19)
=== 読み込みテスト中: spot_summary_2011.parquet ===
  -> OK, shape=(17568, 19)
=== 読み込みテスト中: spot_summary_2012.parquet ===
  -> OK, shape=(17520, 19)
=== 読み込みテスト中: spot_summary_2013.parquet ===
  -> OK, shape=(17520, 19)
=== 読み込みテスト中: spot_summary_2014.parquet ===
  -> OK, shape=(17520, 19)
=== 読み込みテスト中: spot_summary_2015.parquet ===
  -> OK, shape=(17568, 19)
=== 読み込みテスト中: spot_summary_2016.parquet ===
  -> OK, shape=(17520, 19)
=== 読み込みテスト中: spot_summary_2017.parquet ===
  -> OK, shape=(17520, 19)
=== 読み込みテスト中: spot_summary_2018.parquet ===
  -> OK, shape=(17520, 19)
=== 読み

受渡日,時刻コード,売り入札量(kWh),買い入札量(kWh),約定総量(kWh),システムプライス(円/kWh),エリアプライス北海道(円/kWh),エリアプライス東北(円/kWh),エリアプライス東京(円/kWh),エリアプライス中部(円/kWh),エリアプライス北陸(円/kWh),エリアプライス関西(円/kWh),エリアプライス中国(円/kWh),エリアプライス四国(円/kWh),エリアプライス九州(円/kWh),売りブロック入札総量(kWh),売りブロック約定総量(kWh),買いブロック入札総量(kWh),買いブロック約定総量(kWh)
str,i64,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,str,str,str,str
"""2005/04/02""",1,676500,198000,0,6.71,6.71,6.71,"""6.71""",6.71,6.71,6.71,6.71,6.71,6.71,null,null,null,null
"""2005/04/02""",2,676500,192000,0,6.65,6.65,6.65,"""6.65""",6.65,6.65,6.65,6.65,6.65,6.65,null,null,null,null
"""2005/04/02""",3,670000,189000,10000,6.39,6.39,6.39,"""6.39""",6.39,6.39,6.39,6.39,6.39,6.39,null,null,null,null
"""2005/04/02""",4,670000,186500,10000,6.39,6.39,6.39,"""6.39""",6.39,6.39,6.39,6.39,6.39,6.39,null,null,null,null
"""2005/04/02""",5,676500,184500,15000,5.68,5.68,5.68,"""5.68""",5.68,5.68,5.68,5.68,5.68,5.68,null,null,null,null


### Check schema

In [12]:
schema = df_spot.schema

schema

Schema([('受渡日', String),
        ('時刻コード', Int64),
        ('売り入札量(kWh)', Int64),
        ('買い入札量(kWh)', Int64),
        ('約定総量(kWh)', Int64),
        ('システムプライス(円/kWh)', Float64),
        ('エリアプライス北海道(円/kWh)', Float64),
        ('エリアプライス東北(円/kWh)', Float64),
        ('エリアプライス東京(円/kWh)', String),
        ('エリアプライス中部(円/kWh)', Float64),
        ('エリアプライス北陸(円/kWh)', Float64),
        ('エリアプライス関西(円/kWh)', Float64),
        ('エリアプライス中国(円/kWh)', Float64),
        ('エリアプライス四国(円/kWh)', Float64),
        ('エリアプライス九州(円/kWh)', Float64),
        ('売りブロック入札総量(kWh)', String),
        ('売りブロック約定総量(kWh)', String),
        ('買いブロック入札総量(kWh)', String),
        ('買いブロック約定総量(kWh)', String)])

### Null count

In [14]:
null_count = df_spot.null_count()
null_count.with_columns((pl.col(pl.Utf8) > 0).alias("has_nulls"))

受渡日,時刻コード,売り入札量(kWh),買い入札量(kWh),約定総量(kWh),システムプライス(円/kWh),エリアプライス北海道(円/kWh),エリアプライス東北(円/kWh),エリアプライス東京(円/kWh),エリアプライス中部(円/kWh),エリアプライス北陸(円/kWh),エリアプライス関西(円/kWh),エリアプライス中国(円/kWh),エリアプライス四国(円/kWh),エリアプライス九州(円/kWh),売りブロック入札総量(kWh),売りブロック約定総量(kWh),買いブロック入札総量(kWh),買いブロック約定総量(kWh)
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,960,0,3744,0,0,0,0,0,0,138576,138576,213552,213552


### Convert Column Type

In [8]:
for n in df_spot["エリアプライス東京(円/kWh)"].unique():
    try:
        price = float(n)
        #print(price)
    except:
        print(f"Cannnot convert to integer : {n}")

Cannnot convert to integer : None


In [15]:
# Filter out rows where "エリアプライス東京(円/kWh)" is null
df_spot_nonnull = df_spot.filter(pl.col("エリアプライス東京(円/kWh)").is_null())

print("before:", df_spot.shape)
print("after :", df_spot_nonnull.shape)

# show a few rows to inspect
df_spot_nonnull.head()

before: (362208, 19)
after : (3744, 19)


受渡日,時刻コード,売り入札量(kWh),買い入札量(kWh),約定総量(kWh),システムプライス(円/kWh),エリアプライス北海道(円/kWh),エリアプライス東北(円/kWh),エリアプライス東京(円/kWh),エリアプライス中部(円/kWh),エリアプライス北陸(円/kWh),エリアプライス関西(円/kWh),エリアプライス中国(円/kWh),エリアプライス四国(円/kWh),エリアプライス九州(円/kWh),売りブロック入札総量(kWh),売りブロック約定総量(kWh),買いブロック入札総量(kWh),買いブロック約定総量(kWh)
str,i64,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,str,str,str,str
"""2011/03/15""",1,605500,417000,146000,9.27,3.0,3.0,null,9.35,9.35,9.35,9.35,9.35,9.35,null,null,null,null
"""2011/03/15""",2,605500,412000,146000,9.29,3.0,3.0,null,9.38,9.38,9.38,9.38,9.38,9.38,null,null,null,null
"""2011/03/15""",3,605500,407500,145500,9.3,3.0,3.0,null,9.39,9.39,9.39,9.39,9.39,9.39,null,null,null,null
"""2011/03/15""",4,607000,405000,147000,9.37,3.0,3.0,null,9.46,9.46,9.46,9.46,9.46,9.46,null,null,null,null
"""2011/03/15""",5,625500,402000,148500,9.24,3.0,3.0,null,9.34,9.34,9.34,9.34,9.34,9.34,null,null,null,null


In [17]:
df_spot_nonnull["受渡日"].unique().sort()

受渡日
str
"""2011/03/15"""
"""2011/03/16"""
"""2011/03/17"""
"""2011/03/18"""
"""2011/03/19"""
…
"""2011/05/27"""
"""2011/05/28"""
"""2011/05/29"""
